# End-to-End ML Pipeline — Telco Customer Churn Prediction

**Task 2 — AI/ML Engineering Advanced Internship, DevelopersHub Corporation**

**Objective:** Build a reusable, production-ready ML pipeline for predicting
customer churn using scikit-learn's `Pipeline` API, tune hyperparameters
with `GridSearchCV`, and export the complete pipeline with `joblib`.

This notebook calls into the reusable modules under `src/` so the same
logic backs both this notebook, the CLI scripts, and the Streamlit app.
All code in this notebook has been validated end-to-end — see `report.md`
for the actual results.


## 1. Setup

In [ ]:
import sys
sys.path.append("..")  # so `import src...` works when running from notebooks/

import pandas as pd
import json
from src.config import NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET_COL, RESULTS_DIR, MODELS_DIR


## 2. Dataset loading & preprocessing

Loads the Telco churn dataset, cleans it (TotalCharges coercion, dedup,
target encoding), and builds the train/test split.

In [ ]:
from src.preprocessing import load_and_clean, get_feature_target_split, train_test_split_data

df = load_and_clean()
print(f"Rows: {len(df)}  |  Churn rate: {df[TARGET_COL].mean():.1%}")
df.head()


In [ ]:
X, y = get_feature_target_split(df)
X_train, X_test, y_train, y_test = train_test_split_data(X, y)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.1%}  |  Test churn rate: {y_test.mean():.1%}")


## 3. Exploratory look at churn drivers

Quick sanity check before modeling: churn rate by contract type and
internet service (the two features that should matter most).

In [ ]:
pd.concat([df["Contract"], df[TARGET_COL]], axis=1).groupby("Contract")[TARGET_COL].mean().sort_values(ascending=False)


In [ ]:
pd.concat([df["InternetService"], df[TARGET_COL]], axis=1).groupby("InternetService")[TARGET_COL].mean().sort_values(ascending=False)


## 4. Building the pipeline

`ColumnTransformer` (median-impute + scale numeric, most-frequent-impute +
one-hot categorical) wrapped with a classifier in a single `Pipeline` —
this is what makes the exported artifact self-contained.

In [ ]:
from src.pipeline import build_pipeline

lr_pipeline = build_pipeline("logistic_regression")
lr_pipeline


## 5. Hyperparameter tuning with GridSearchCV

Tunes both Logistic Regression and Random Forest, 5-fold CV, optimizing F1
(not accuracy — with a 30/70 class split, accuracy alone would reward a
model that just predicts "No churn" for everyone).

In [ ]:
from src.train import tune_model
from src.config import PARAM_GRIDS

fitted_grids = {}
for model_name in PARAM_GRIDS:
    grid, elapsed = tune_model(model_name, X_train, y_train)
    fitted_grids[model_name] = grid


## 6. Exporting the winning pipeline with joblib

In [ ]:
import joblib

winner_name = max(fitted_grids, key=lambda k: fitted_grids[k].best_score_)
print(f"Winner: {winner_name} (CV F1 = {fitted_grids[winner_name].best_score_:.4f})")

joblib.dump(fitted_grids[winner_name].best_estimator_, MODELS_DIR / "churn_pipeline.joblib")
for name, grid in fitted_grids.items():
    joblib.dump(grid.best_estimator_, MODELS_DIR / f"{name}_pipeline.joblib")

X_test.to_csv(RESULTS_DIR / "X_test.csv", index=False)
y_test.to_csv(RESULTS_DIR / "y_test.csv", index=False)
print("Exported.")


## 7. Full evaluation on the held-out test set

In [ ]:
from src.evaluate import main as evaluate_main
all_metrics = evaluate_main()


## 8. Visualizations

In [ ]:
from src.visualize import generate_all_plots
generate_all_plots()


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(RESULTS_DIR / "figures" / "metric_comparison.png")))


In [ ]:
display(Image(filename=str(RESULTS_DIR / "figures" / "confusion_matrices.png")))
display(Image(filename=str(RESULTS_DIR / "figures" / "roc_curves.png")))


In [ ]:
display(Image(filename=str(RESULTS_DIR / "figures" / f"feature_importance_{winner_name}.png")))


## 9. Try a prediction on a new customer

In [ ]:
from src.predict import predict_one

at_risk_customer = {
    "tenure": 3, "MonthlyCharges": 95.0, "TotalCharges": 285.0,
    "gender": "Female", "SeniorCitizen": "0", "Partner": "No", "Dependents": "No",
    "PhoneService": "Yes", "MultipleLines": "Yes", "InternetService": "Fiber optic",
    "OnlineSecurity": "No", "OnlineBackup": "No", "DeviceProtection": "No",
    "TechSupport": "No", "StreamingTV": "Yes", "StreamingMovies": "Yes",
    "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
}
predict_one(at_risk_customer)


In [ ]:
loyal_customer = {**at_risk_customer, "tenure": 60, "Contract": "Two year",
                   "TechSupport": "Yes", "OnlineSecurity": "Yes", "PaymentMethod": "Credit card (automatic)"}
predict_one(loyal_customer)


## 10. Summary & Insights

- **Logistic Regression (CV F1 = 0.6404) narrowly beat Random Forest
  (CV F1 = 0.6321)** on this dataset — with ~4,000 rows and largely
  additive, categorical churn drivers, the simpler linear model
  generalizes at least as well as the more flexible tree ensemble, and
  trains ~90x faster (1.6s vs 146.8s).
- **Recall matters more than raw accuracy for churn prediction**: Logistic
  Regression's `class_weight="balanced"` setting trades some precision for
  much higher recall (79.3% vs Random Forest's 66.4%) — catching more
  actual churners is worth more to the business than avoiding false alarms,
  since a missed churner is a fully lost customer.
- **Contract type and internet service type are the dominant predictors**,
  consistent with real-world churn research: month-to-month contracts have
  zero switching cost, while long-term contracts and bundled services
  create retention lock-in.
- The exported `models/churn_pipeline.joblib` is fully self-contained —
  it can classify a brand-new, raw customer record with no manual
  preprocessing step required by the caller.

See `report.md` for the full write-up and `README.md` for setup
instructions. Try the live version with `streamlit run app/streamlit_app.py`.